In [4]:
# --- 1. 环境准备 (与您提供的代码相同) ---
import os
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory
)
from dotenv import load_dotenv

# 加载.env文件中的环境变量
load_dotenv()

# 设置OPENAI API的基础URL和API密钥
# 请确保你的 .env 文件中有 OPENAI_API_BASE 和 OPENAI_API_KEY
# 示例 .env 文件内容:
# OPENAI_API_BASE="https://api.example.com/v1"
# OPENAI_API_KEY="sk-xxxxxxxxxx"
os.environ["OPENAI_API_BASE"] = os.getenv("OPENAI_API_BASE")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# --- 2. 初始化LLM ---
# 为了清晰展示，我们使用同一个llm实例
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- 3. 定义一个通用的测试函数来展示区别 ---
def run_memory_test(memory, memory_name):
    """
    运行一个标准对话流程来测试给定的记忆模块。
    
    Args:
        memory: 要测试的记忆对象实例。
        memory_name: 记忆模块的名称（用于打印）。
    """
    print(f"=============== 开始测试: {memory_name} ===============")
    
    # 创建对话链，verbose=True可以让我们看到完整的提示，理解记忆是如何工作的
    chain = ConversationChain(llm=llm, memory=memory, verbose=True)

    # --- 进行一个包含多个信息点的对话 ---
    print("\n[第一轮对话]")
    chain.predict(input="你好，我叫李华。")
    
    print("\n[第二轮对话]")
    chain.predict(input="我住在东京。")

    print("\n[第三轮对话]")
    chain.predict(input="我喜欢踢足球。")

    print("\n[第四轮对话 - 测试记忆]")
    # 这个问题需要模型回忆起之前所有轮次的信息
    final_response = chain.predict(input="我是谁？我住在哪？我的爱好是什么？")
    
    print("\n----------- 测试结果 -----------")
    print(f"模型最终回答: {final_response}")
    
    # 打印记忆的最终状态，这是理解其工作原理的关键
    print("\n----------- 记忆最终状态 -----------")
    # 使用 .load_memory_variables({}) 来获取记忆的格式化内容
    print(memory.load_memory_variables({}))
    print(f"=============== 结束测试: {memory_name} ===============\n\n")


# --- 4. 依次测试三种记忆模式 ---
# 完整记忆: 应该能记住所有内容
buffer_memory = ConversationBufferMemory()
run_memory_test(buffer_memory, "ConversationBufferMemory (完整记忆)")

# 窗口记忆 (k=1): 应该只记得最后一轮对话（爱好），而忘记了名字和地点
window_memory = ConversationBufferWindowMemory(k=1)
run_memory_test(window_memory, "ConversationBufferWindowMemory (窗口记忆, k=1)")

# 摘要记忆: 应该能通过摘要记住所有关键信息
summary_memory = ConversationSummaryMemory(llm=llm)
run_memory_test(summary_memory, "ConversationSummaryMemory (摘要记忆)")

=============== 开始测试: ConversationBufferMemory (完整记忆) ===============

[第一轮对话]


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: 你好，我叫李华。
AI:

> Finished chain.

[第二轮对话]


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: 你好，我叫李华。
AI: 你好，李华！很高兴认识你。我是一个人工智能助手，可以帮助你回答问题或提供信息。你今天过得怎么样？有什么我可以帮忙的吗？
Human: 我住在东京。
AI:

> Finished chain.

[第三轮对话]


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly con